# 02｜IoU：怎样衡量两个边界框有多接近

上一课已经知道，目标检测模型要为每个物体预测类别和边界框。

现在出现了一个最直接的问题：模型预测了一个框，真实标注也有一个框，怎样客观判断它们有多接近？

这一课只学习 IoU。我们会从两个矩形的交集与并集出发，完整算一遍，不提前进入 NMS、Anchor、YOLO 或 DETR。

## 1. 为什么不能只看两个中心点的距离

假设预测框的中心点与真实框中心点完全重合，这个预测就一定好吗？

不一定。预测框可能非常大，把整张图片都框住；也可能非常小，只覆盖物体中心的一小块。它们的中心点相同，但大小明显不正确。

反过来，两个框的宽高相同，也不代表预测正确。如果预测框整体向右偏移，它们覆盖的区域仍然不同。

所以衡量边界框质量时，需要同时考虑：

- 位置是否接近。
- 大小是否接近。
- 两个框实际覆盖了多少共同区域。

IoU 正是从“共同覆盖区域”出发设计的。

## 2. 先理解交集与并集

设预测框为 $A$，真实框为 $B$。

### 交集

$A\cap B$ 表示两个框共同覆盖的区域。重叠得越多，交集面积通常越大。

### 并集

$A\cup B$ 表示至少被其中一个框覆盖的全部区域。它包括：

- 只被 $A$ 覆盖的部分。
- 只被 $B$ 覆盖的部分。
- 两个框共同覆盖的部分。

IoU 比较的不是交集面积本身，而是交集在并集里所占的比例。

## 3. IoU 的定义

IoU 的全称是 Intersection over Union，中文通常译为交并比。定义为：

$$
\operatorname{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}
$$

其中：

- $|A\cap B|$：两个框的交集面积。
- $|A\cup B|$：两个框的并集面积。

用一句话理解：

> 两个框共同覆盖的面积，占它们总覆盖面积的多少。

## 4. IoU 的取值范围

IoU 的取值范围是：

$$
0\leq\operatorname{IoU}(A,B)\leq1
$$

可以先通过三个极端情况理解：

| 两个框的关系 | 交集 | IoU |
|---|---:|---:|
| 完全不重叠 | 0 | 0 |
| 部分重叠 | 大于 0、小于并集 | 0 到 1 之间 |
| 完全重合 | 等于并集 | 1 |

因此，IoU 越接近 1，两个框越相似；越接近 0，两个框的空间重叠越少。

## 5. 计算前先统一使用 `xyxy`

上一课学习了两种边界框格式。计算交集时，`xyxy` 更直观：

$$
A=(x_{1}^{A},y_{1}^{A},x_{2}^{A},y_{2}^{A})
$$

$$
B=(x_{1}^{B},y_{1}^{B},x_{2}^{B},y_{2}^{B})
$$

其中：

- 下标 1 表示左上角。
- 下标 2 表示右下角。
- 上标 $A$ 或 $B$ 表示坐标属于哪个框。

如果手中是 `cxcywh`，可以先转换成 `xyxy`，再计算交集。

## 6. 第一步：确定交集矩形的四条边

两个矩形的交集仍然是一个矩形，前提是它们确实重叠。

交集的左边界要取两个左边界中更靠右的那个：

$$
x_{1}^{I}=\max(x_{1}^{A},x_{1}^{B})
$$

交集的上边界要取两个上边界中更靠下的那个：

$$
y_{1}^{I}=\max(y_{1}^{A},y_{1}^{B})
$$

交集的右边界要取两个右边界中更靠左的那个：

$$
x_{2}^{I}=\min(x_{2}^{A},x_{2}^{B})
$$

交集的下边界要取两个下边界中更靠上的那个：

$$
y_{2}^{I}=\min(y_{2}^{A},y_{2}^{B})
$$

这里的核心不是死记 `max` 和 `min`，而是想象：交集必须同时待在两个框的内部，因此它的边界只能向共同区域收缩。

## 7. 第二步：计算交集宽、高和面积

确定交集边界后，交集宽度和高度为：

$$
w_I=\max(0,x_{2}^{I}-x_{1}^{I})
$$

$$
h_I=\max(0,y_{2}^{I}-y_{1}^{I})
$$

交集面积为：

$$
S_I=w_Ih_I
$$

为什么一定要与 0 取最大值？

如果两个框完全分开，可能出现 $x_{2}^{I}<x_{1}^{I}$。直接相减会得到负宽度，但几何面积不可能是负数。因此要把负值截断为 0。

只要交集宽度或高度有一个为 0，交集面积就是 0。

## 8. 第三步：分别计算两个框的面积

框 $A$ 的宽、高和面积为：

$$
\begin{aligned}
w_A&=x_{2}^{A}-x_{1}^{A}\\
h_A&=y_{2}^{A}-y_{1}^{A}\\
S_A&=w_Ah_A
\end{aligned}
$$

框 $B$ 同理：

$$
\begin{aligned}
w_B&=x_{2}^{B}-x_{1}^{B}\\
h_B&=y_{2}^{B}-y_{1}^{B}\\
S_B&=w_Bh_B
\end{aligned}
$$

正常边界框应满足右边界不小于左边界、下边界不小于上边界。数据标注或模型输出若不满足这个条件，需要先检查或修正框的表示。

## 9. 第四步：计算并集面积

最容易犯错的地方是直接把两个框面积相加：

$$
S_A+S_B
$$

这样会把交集区域计算两次：一次包含在 $S_A$ 中，一次包含在 $S_B$ 中。

所以并集面积必须减去一次交集面积：

$$
S_U=S_A+S_B-S_I
$$

最后：

$$
\operatorname{IoU}=\frac{S_I}{S_U}
$$

这一步的关键是理解“加两次、减一次”，而不是只背公式。

## 10. 用具体数字完整计算一次

设两个框为：

$$
A=(100,100,300,300)
$$

$$
B=(200,150,400,350)
$$

它们都采用 `xyxy` 格式。先求交集边界：

$$
\begin{aligned}
x_{1}^{I}&=\max(100,200)=200\\
y_{1}^{I}&=\max(100,150)=150\\
x_{2}^{I}&=\min(300,400)=300\\
y_{2}^{I}&=\min(300,350)=300
\end{aligned}
$$

因此，交集宽、高和面积为：

$$
\begin{aligned}
w_I&=300-200=100\\
h_I&=300-150=150\\
S_I&=100\times150=15000
\end{aligned}
$$

## 11. 接着算两个框与并集面积

框 $A$ 的面积：

$$
S_A=(300-100)(300-100)=200\times200=40000
$$

框 $B$ 的面积：

$$
S_B=(400-200)(350-150)=200\times200=40000
$$

并集面积：

$$
S_U=40000+40000-15000=65000
$$

最终 IoU：

$$
\operatorname{IoU}=\frac{15000}{65000}=\frac{3}{13}\approx0.2308
$$

也就是说，两个框共同覆盖的区域约占总覆盖区域的 23.08%。它们有一定重叠，但并不十分接近。

## 12. 四种特殊情况

### 情况一：完全重合

两个框相同时，交集等于并集，因此 IoU 为 1。

### 情况二：完全分离

两个框没有任何共同区域，交集面积为 0，因此 IoU 为 0。

### 情况三：边缘刚好接触

两个框虽然碰到了，但没有形成有面积的共同区域。交集宽度或高度为 0，因此 IoU 仍然为 0。

### 情况四：一个框完全包含另一个框

设小框完全位于大框内部，则交集就是小框，并集就是大框：

$$
\operatorname{IoU}=\frac{S_{small}}{S_{large}}
$$

这说明中心点和物体位置可能都对，但如果预测框远大于真实框，IoU 仍然不会很高。

## 13. IoU 阈值是什么意思

实际任务中，经常用一个阈值判断两个框是否足够接近。例如暂时设阈值为 0.5：

$$
\begin{cases}
\operatorname{IoU}\geq0.5,&\text{认为重叠程度达到当前要求}\\
\operatorname{IoU}<0.5,&\text{认为重叠程度没有达到当前要求}
\end{cases}
$$

但 0.5 不是宇宙统一标准。不同数据集、训练分配规则、NMS 设置和评价协议会使用不同阈值，有些评价还会综合多个阈值。

现在只需要记住：**阈值是人为规定的合格线，IoU 才是两个框实际计算出的重叠度。**

## 14. IoU 不等于置信度，也不判断类别

IoU、类别概率和置信度回答的是不同问题：

| 数值 | 回答的问题 | 是否直接比较两个框的几何关系 |
|---|---|---|
| IoU | 两个框重叠得有多好？ | 是 |
| 类别概率 | 框中的物体属于某类别的可能性多大？ | 否 |
| 置信度或目标性分数 | 模型多相信这里存在有效目标？ | 否 |

可能出现：

- 类别预测正确，但框的位置很差，IoU 很低。
- 框与真实目标高度重合，但类别预测错误。
- 模型置信度很高，但实际边界框不准确。

所以完整检测评价不能只看其中一个数。

## 15. IoU 后面会出现在哪里

虽然这一课不展开后续算法，但可以先建立地图。IoU 常在三个阶段出现：

### 训练前或训练中：目标分配

判断某个候选框与哪个真实框更接近，从而决定它应该学习哪个目标。

### 推理后处理：NMS

判断两个高置信度预测框是否过度重叠，从而识别重复检测。

### 模型评价：检测是否命中

判断预测框与真实框是否达到评价协议要求的重叠程度。

具体规则并不总是相同，但三处都使用了“两个框有多重叠”这个几何信息。

## 16. IoU 的一个重要局限

只要两个框不重叠，IoU 都是 0。

问题是，不重叠的框之间仍然可能有明显差别：

- 预测框距离真实框只有 1 个像素。
- 预测框位于图片的另一个角落。

它们的 IoU 都是 0，普通 IoU 无法继续告诉我们哪一个更接近。

这也是为什么后续会出现 GIoU、DIoU、CIoU 等扩展指标或损失。DETR 中会遇到 GIoU，但现在不展开公式。先把普通 IoU 真正掌握。

## 17. 关于像素边界的一个说明

本课程按现代深度学习框检测中常见的连续坐标几何理解边界框：

$$
w=x_{max}-x_{min},\qquad h=y_{max}-y_{min}
$$

有些较旧的图像处理代码把坐标理解为包含两端的离散像素索引，因此面积计算中可能出现加 1。

两种约定不能混用。学习某个数据集或代码库时，必须确认坐标表示的定义。本阶段统一使用不加 1 的连续坐标约定。

## 18. 常见错误

### 错误一：把并集写成两个面积直接相加

交集会被重复计算，正确做法是再减去一次交集面积。

### 错误二：交集宽高出现负数后继续相乘

两个负数相乘甚至会得到正面积。必须分别使用 $\max(0,\cdot)$ 截断。

### 错误三：混用 `xyxy` 和 `cxcywh`

四个数字看起来相同，但含义不同。计算前必须确认格式。

### 错误四：认为 IoU 高就代表检测完全正确

IoU 只衡量框的几何重叠，不判断类别是否正确。

### 错误五：把阈值 0.5 当成固定规则

阈值取决于具体用途和评价协议。

## 19. 本节小结

这一课需要真正记住六个结论：

1. IoU 衡量两个边界框共同覆盖的面积占总覆盖面积的比例。
2. IoU 等于交集面积除以并集面积，取值范围为 0 到 1。
3. 计算交集时，左上角取较大的坐标，右下角取较小的坐标。
4. 交集宽和高必须分别与 0 取最大值，避免产生负面积。
5. 并集面积等于两个框面积之和减去一次交集面积。
6. IoU 只衡量几何重叠，不代表类别正确，也不等于模型置信度。

完整计算路线是：

$$
\text{两个 }xyxy\text{ 框}
\rightarrow\text{交集边界}
\rightarrow\text{交集面积}
\rightarrow\text{并集面积}
\rightarrow\operatorname{IoU}
$$

下一课将讨论：模型为什么不能只预测类别，还必须学习四个连续边界框坐标，以及“分类”和“边界框回归”怎样组成一条检测结果。

## 20. 自测问题

1. 为什么只比较两个框的中心点距离不够？
2. IoU 的分子和分母分别是什么？
3. IoU 的取值范围是什么？
4. 两个框完全重合时，IoU 是多少？
5. 两个框只有边缘接触时，IoU 为什么仍然为 0？
6. 为什么求交集左边界时使用 `max`？
7. 为什么求交集右边界时使用 `min`？
8. 为什么交集宽和高要分别与 0 取最大值？
9. 为什么并集面积不是简单的 $S_A+S_B$？
10. 框 $A$ 与框 $B$ 的面积都是 100，交集面积是 40，并集面积和 IoU 分别是多少？
11. 一个小框完全位于大框内部时，交集和并集分别是谁的面积？
12. IoU 为 0.8 是否能证明类别预测正确？
13. IoU 阈值 0.5 是否适用于所有场景？
14. 普通 IoU 对完全不重叠的框有什么局限？

### 自测参考答案

1. 中心相同的框可能大小差异很大，大小相同的框也可能整体错位。
2. 分子是交集面积，分母是并集面积。
3. 0 到 1。
4. 1。
5. 接触线没有面积，交集宽度或高度为 0。
6. 交集左边界必须同时位于两个框内部，因此要选择更靠右的左边界。
7. 交集右边界必须同时位于两个框内部，因此要选择更靠左的右边界。
8. 两框分离时坐标差可能为负，但几何宽度、高度和面积不能为负。
9. 直接相加会把交集计算两次，所以要减去一次交集面积。
10. 并集为 $100+100-40=160$，IoU 为 $40/160=0.25$。
11. 交集是小框面积，并集是大框面积。
12. 不能。IoU 只表示几何重叠度。
13. 不是，阈值取决于具体算法用途和评价协议。
14. 所有不重叠情况都得到 0，无法继续区分两个框相距多远。